# OOD 결과 테이블 생성

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 📋 실행 순서

### Option A: 이미 CSV 파일이 있는 경우 (권장)
1. **셀 1-6**을 순서대로 실행

### Option B: WandB에서 최신 데이터를 가져오는 경우
1. **셀 0-1, 0-2, 0-3**을 실행하여 WandB에서 데이터 수집
   - ⚠️ **주의**: 시간이 오래 걸릴 수 있습니다 (1-5분)
   - `results/finished_runs_data.csv` 파일이 생성/업데이트됩니다
2. **셀 1-6**을 순서대로 실행

---

## 📑 셀별 역할 요약

| 셀 | 역할 |
|----|------|
| **0-1** | WandB API 초기화, project_name, sweep_id 설정 |
| **0-2** | WandB에서 run 데이터 수집 (config, summary, history), Task/forgetting 계산, runs_df 생성 |
| **0-2 결과** | runs_df.head()로 수집 결과 미리보기 |
| **분포 확인** | fusion_type, increment 고유값 출력 |
| **0-3** | runs_df를 CSV로 저장 (all_runs, finished_runs) |
| **1. 설정** | preset 선택 → cfg (datasets, increments, fusion_type, cl_methods 등) 설정 |
| **1. 로드** | CSV에서 df 로드 |
| **2. 파싱** | sweep_name에서 model, modality, dataset 추출 |
| **3. 함수** | create_ood_result_table, save_ood_table_to_excel 정의 |
| **4. 생성** | cfg 기반으로 테이블 생성 및 엑셀 저장 |
| **진단** | cfg 조건에 맞는 데이터 존재 여부 확인 |

---

## 0. WandB에서 데이터 수집 (선택사항)

**이미 `results/finished_runs_data.csv` 파일이 있다면 이 섹션을 건너뛰고 셀 1부터 시작하세요!**

### 💡 특정 Sweep만 불러오기
셀 0-1에서 `sweep_id`를 지정하면 해당 sweep의 runs만 불러옵니다 (전체 프로젝트보다 훨씬 빠름).
- 예: `sweep_id = "abc123xy"`
- Sweep ID는 WandB 웹 UI의 sweep URL에서 확인 가능합니다.


In [1]:
# =============================================================================
# [셀 0-1] WandB API 초기화
# =============================================================================
# • wandb.Api(): WandB 클라우드에 연결하여 프로젝트/run 데이터를 가져올 수 있는 API 객체 생성
# • project_name: "entity/project" 형식. WandB 웹에서 프로젝트 URL 확인 가능
# • sweep_id: 특정 sweep만 불러올 때 사용. None이면 프로젝트 전체 runs 조회 (느림)
#    - sweep_id 지정 시 해당 sweep의 runs만 가져와 훨씬 빠름
#    - Sweep ID는 WandB sweep 페이지 URL의 마지막 부분 (예: .../sweeps/abc123xy)
# =============================================================================

import wandb
from pathlib import Path

api = wandb.Api()
project_name = "mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)"

# 특정 sweep만 불러오려면 sweep_id를 지정하세요. None이면 프로젝트 전체 runs를 불러옵니다.
'''
 - gpubp5iu (λ=0.3, β=0.02)
 - z1r3t748 (λ=0.5, β=0.1)
 - sr32tgjw (λ=0.5, β=0.01) 
'''
sweep_id = "xx5mgvsg"  # 예: "abc123xy"  → 해당 sweep의 runs만 불러옴

print(f"📊 연결된 프로젝트: {project_name}")
print(f"📌 Sweep ID: {sweep_id if sweep_id else '(전체 프로젝트)'}")
print("✅ WandB API 초기화 완료")


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /workspace/.netrc.


📊 연결된 프로젝트: mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)
📌 Sweep ID: xx5mgvsg
✅ WandB API 초기화 완료


In [2]:
# =============================================================================
# [셀 0-2] WandB에서 Run 데이터 수집
# =============================================================================
# 이 셀은 다음을 수행합니다:
#
# 1. _get_task_acc_cols(): history 컬럼 중 Task/[XX-YY]_acc 패턴 추출
#    - WandB에 Task/[00-01]_acc, Task/[02-03]_acc 등으로 기록됨
#    - task 번호 순으로 정렬하여 반환
#
# 2. _get_history_data(): run.history() 또는 scan_history()로 step별 기록 가져오기
#    - summary에는 Task/[xx-xx]_acc가 NaN인 경우가 있어, history의 마지막 행 사용
#
# 3. _compute_forgetting(): CL forgetting 지표 계산 (mammoth 방식)
#    - 각 task의 max(과거 acc) - 최종 acc를 평균
#
# 4. get_all_runs_from_project(): 각 run의 config + summary + history에서
#    Task/[xx-xx]_acc 채우기 + forgetting 계산 → runs_data 리스트로 반환
#
# 5. runs_df: 전체 데이터를 DataFrame으로 생성 (다음 셀에서 저장/분석에 사용)
# =============================================================================

import pandas as pd
import numpy as np
import re

def _get_task_acc_cols(hist_or_row):
    """Task/[XX-YY]_acc 패턴 컬럼명 추출 (history 컬럼 또는 dict keys)"""
    keys = hist_or_row.columns if hasattr(hist_or_row, 'columns') else hist_or_row.keys()
    acc = [k for k in keys if isinstance(k, str) and k.startswith('Task/[') and k.endswith(']_acc') and 'NME' not in k]
    return sorted(acc, key=lambda x: int(re.search(r'[(\[]?(\d+)-', str(x)).group(1)) if re.search(r'[(\[]?(\d+)-', str(x)) else 0)

def _get_history_data(run):
    """run.history() 또는 scan_history()로 데이터 수집. (hist, acc_cols) 반환"""
    try:
        hist = run.history(samples=5000)
        if hist is not None and len(hist) >= 1:
            acc_cols = _get_task_acc_cols(hist)
            if acc_cols:
                return hist, acc_cols
    except Exception:
        pass
    try:
        rows = list(run.scan_history())
        if rows:
            acc_cols = _get_task_acc_cols(rows[0])
            if acc_cols:
                return pd.DataFrame(rows), acc_cols
    except Exception:
        pass
    return None, []

def _compute_forgetting(hist, acc_cols):
    """Mammoth-style average forgetting (see 5.paired_ttest / 1.performance_comp_table_tiny forgetting())."""
    if hist is None or len(acc_cols) < 2:
        return np.nan
    n_tasks = len(acc_cols)
    if 'Task/Task_ID' in hist.columns:
        h = hist.dropna(subset=['Task/Task_ID']).copy()
        if h.empty:
            return np.nan
        h = h.sort_values('Task/Task_ID').groupby(
            'Task/Task_ID', sort=False, as_index=False
        ).last()
        h = h.sort_values('Task/Task_ID')
    else:
        h = hist.dropna(subset=[acc_cols[0]], how='all').tail(n_tasks)
    if len(h) < 2 or len(h) < n_tasks:
        return np.nan
    h = h.iloc[:n_tasks]
    results = []
    for t, (_, row) in enumerate(h.iterrows()):
        vals = [
            float(row[c]) if pd.notna(row.get(c)) else 0.0
            for c in acc_cols[: t + 1]
        ]
        results.append(vals + [0.0] * (n_tasks - len(vals)))
    np_res = np.array(results)
    max_acc = np.max(np_res, axis=0)
    return float(
        np.mean([max_acc[i] - results[-1][i] for i in range(n_tasks - 1)])
    )

def get_all_runs_from_project(project_name, sweep_id=None, filters=None, compute_forgetting=True, use_history_for_task_acc=True):
    """프로젝트의 Run 데이터를 가져옵니다.
    
    Args:
        project_name: "entity/project" 형식의 프로젝트 경로
        sweep_id: 특정 sweep의 ID. 지정하면 해당 sweep의 runs만 불러옴. None이면 프로젝트 전체.
        filters: api.runs()에 전달할 추가 필터 (sweep_id 없을 때만 사용)
        compute_forgetting: True이면 각 run의 history에서 Forgetting 계산
    """
    api = wandb.Api()
    
    if sweep_id:
        sweep_path = f"{project_name}/{sweep_id}"
        print(f"🔍 Sweep '{sweep_path}'에서 Run 데이터를 가져오는 중...")
        sweep = api.sweep(sweep_path)
        runs = sweep.runs
    else:
        runs = api.runs(project_name, filters=filters) if filters else api.runs(project_name)
        print(f"🔍 프로젝트 전체 Run 데이터를 가져오는 중...")
    
    runs_data = []
    for i, run in enumerate(runs, 1):
        if i % 10 == 0:
            print(f"   진행중: {i} runs 처리...")
        
        run_info = {
            'run_id': run.id,
            'run_name': run.name,
            'state': run.state,
            'sweep_id': run.sweep.id if run.sweep else None,
            'sweep_name': run.sweep.name if run.sweep else None,
            'created_at': run.created_at,
        }
        run_info.update(run.config)
        run_info.update(run.summary._json_dict)
        hist, acc_cols = _get_history_data(run)
        if use_history_for_task_acc and hist is not None and acc_cols:
            last_row = hist.dropna(subset=acc_cols, how='all').tail(1)
            if len(last_row) > 0:
                for c in acc_cols:
                    v = last_row[c].iloc[0]
                    if pd.notna(v):
                        run_info[c] = float(v)
        if compute_forgetting:
            run_info['Task/forgetting'] = _compute_forgetting(hist, acc_cols)
        runs_data.append(run_info)
    
    print(f"✅ 총 {len(runs_data)}개의 Run 데이터 수집 완료")
    return runs_data

# 데이터 수집 실행
print("=" * 100)
print("🔄 WandB에서 데이터 수집 시작")
print("=" * 100)
if sweep_id:
    print(f"📌 특정 Sweep만 수집: {sweep_id}")
else:
    print("⚠️ 시간이 다소 걸릴 수 있습니다 (run 개수에 따라 1-5분)")
print()

all_runs_data = get_all_runs_from_project(project_name, sweep_id=sweep_id)
runs_df = pd.DataFrame(all_runs_data)

print(f"\n" + "=" * 100)
print(f"📊 데이터 수집 완료")
print("=" * 100)
print(f"총 Run 개수: {len(runs_df)}")
print(f"총 컬럼 개수: {len(runs_df.columns)}")


🔄 WandB에서 데이터 수집 시작
📌 특정 Sweep만 수집: xx5mgvsg

🔍 Sweep 'mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)/xx5mgvsg'에서 Run 데이터를 가져오는 중...
   진행중: 10 runs 처리...
✅ 총 15개의 Run 데이터 수집 완료

📊 데이터 수집 완료
총 Run 개수: 15
총 컬럼 개수: 240


In [3]:
# =============================================================================
# [셀 0-2 결과 확인] 수집된 runs_df 미리보기
# =============================================================================
# • runs_df: 0-2 셀에서 수집한 모든 run의 config, summary, Task/[xx-xx]_acc, forgetting 등
# • head(): 처음 5행만 출력하여 컬럼 구성과 값 분포 확인
# =============================================================================
runs_df.head()

,run_id,run_name,state,sweep_id,sweep_name,created_at,lr,arch,host,mode,...,Task/[12-13]_acc,Task/[14-15]_acc,Task/[16-17]_acc,Task/[18-19]_acc,Task/[20-21]_acc,Task/[22-23]_acc,Task/[24-25]_acc,Task/[26-27]_acc,Task/[28-29]_acc,Task/[30-31]_acc
0,a9ad5887,tbn_mand_wo_morst_a9ad5887,finished,xx5mgvsg,Evaluate Ablation (w/o MoRST) TBN-MAND (All) M...,2026-04-01T05:23:04Z,0.001,BNInception,ee60c5c7682d,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4b60d4ff,tbn_mand_wo_morst_4b60d4ff,finished,xx5mgvsg,Evaluate Ablation (w/o MoRST) TBN-MAND (All) M...,2026-04-01T05:23:14Z,0.001,BNInception,ee60c5c7682d,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,e6b8fa7b,tbn_mand_wo_morst_e6b8fa7b,finished,xx5mgvsg,Evaluate Ablation (w/o MoRST) TBN-MAND (All) M...,2026-04-01T05:23:24Z,0.001,BNInception,ee60c5c7682d,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02a4416a,tbn_mand_wo_morst_02a4416a,finished,xx5mgvsg,Evaluate Ablation (w/o MoRST) TBN-MAND (All) M...,2026-04-01T05:23:34Z,0.001,BNInception,ee60c5c7682d,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,dd27250a,tbn_mand_wo_morst_dd27250a,finished,xx5mgvsg,Evaluate Ablation (w/o MoRST) TBN-MAND (All) M...,2026-04-01T05:23:45Z,0.001,BNInception,ee60c5c7682d,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# =============================================================================
# [데이터 분포 확인] fusion_type, increment 고유값 출력
# =============================================================================
# • fusion_type: 모달리티 결합 방식 (예: mand_fusion, concat, auxiliary_head 등)
# • increment: CL task 수 (예: 10, 5, 2 → 데이터셋을 10/5/2개 구간으로 나눔)
# → 이후 테이블 생성 시 어떤 fusion_type, increment 조합을 사용할지 참고
# =============================================================================
print("1. fusion_type: ", runs_df["fusion_type"].unique())
print("2. increment: ", runs_df["increment"].unique())

1. fusion_type:  ['mand_fusion']
2. increment:  [8 4 2]


In [5]:
# =============================================================================
# [셀 0-3] 수집한 데이터를 CSV 파일로 저장
# =============================================================================
# • output_dir: notebook/results/ (노트북 위치 기준)
# • all_runs_file: 전체 run (running 포함) → all_runs_data{sweep_id}.csv
# • finished_runs_file: state=='finished'인 run만 → finished_runs_data{sweep_id}.csv
# • sweep_id가 있으면 파일명에 sweep_id 붙여 여러 sweep 결과를 분리 저장
# • finished_runs_filename: 다음 섹션(데이터 로드)에서 사용할 파일명 변수
# =============================================================================

import os
# 노트북 파일이 있는 디렉토리를 기준으로 경로 설정
notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'
output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

# sweep_id가 있으면 sweep별로 파일명 구분 (여러 sweep 결과를 보관 가능)
suffix = f"_{sweep_id}" if sweep_id else ""
all_runs_file = f"all_runs_data{suffix}.csv"
finished_runs_file = f"finished_runs_data{suffix}.csv"

# 전체 데이터 저장
output_file = output_dir / all_runs_file
runs_df.to_csv(output_file, index=False)

# 완료된 run만 필터링하여 저장
finished_runs = runs_df[runs_df['state'] == 'finished']

print("=" * 100)
print("💾 CSV 파일 저장 완료")
print("=" * 100)
print(f"✅ 전체 데이터: {output_file}")
print(f"   - 총 Run 개수: {len(runs_df)}")
print(f"   - 파일 크기: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
if sweep_id:
    print(f"   - Sweep ID: {sweep_id}")

if len(finished_runs) > 0:
    finished_output = output_dir / finished_runs_file
    finished_runs.to_csv(finished_output, index=False)
    print(f"\n✅ 완료된 Run 파일: {finished_output}")
    print(f"   - 완료된 Run 개수: {len(finished_runs)}")
else:
    print(f"\n⚠️ 완료된(finished) Run이 없습니다. 아래 테이블 생성 시 먼저 완료된 Run을 수집하세요.")

# 다음 셀(데이터 로드)에서 사용할 파일명
finished_runs_filename = finished_runs_file

print("\n✨ 데이터 수집 및 저장 완료!")
print("👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.")


💾 CSV 파일 저장 완료
✅ 전체 데이터: /workspace/MMEA-MoAS/notebook/results/all_runs_data_xx5mgvsg.csv
   - 총 Run 개수: 15
   - 파일 크기: 0.04 MB
   - Sweep ID: xx5mgvsg

✅ 완료된 Run 파일: /workspace/MMEA-MoAS/notebook/results/finished_runs_data_xx5mgvsg.csv
   - 완료된 Run 개수: 15

✨ 데이터 수집 및 저장 완료!
👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.


## 1. 분석 설정 (동적 변경 가능)

아래 설정 셀에서 **preset** 또는 **수동 설정**을 변경한 뒤 데이터 로드 → 파싱 → 테이블 생성을 실행하세요.

**preset 예시 (UESTC-MMEA-CL / eval 스윕):**
- `preset = "mmea_mand"` → `scripts-eval/TBN/All/1.uestc-mmea-mand-herding.yaml` 계열 (TBN-MAND, `mand_fusion`, dataset 문자열: **UESTC-MMEA-CL**)
- `preset = None` → `PRESETS["mmea_mand"]` 기본값

*(DataEgo 실험용 preset은 아래 PRESETS 블록에서 주석 해제)*

In [6]:
# =============================================================================
# [셀 1. 분석 설정] preset 또는 수동 설정
# =============================================================================
# • preset: "mmea_mand"          → UESTC-MMEA-CL MAND eval 스윕 (1.uestc-mmea-mand-herding.yaml)
#           "mmea_mand_wo_morst" → UESTC-MMEA-CL Ablation (w/o MoRST) eval 스윕 (2.uestc-mmea-mand-wo-morst-herding.yaml)
#           None                 → PRESETS["mmea_mand"] 기본값
#
# • PRESETS
#   - target_datasets: WandB sweep_name 안의 문자열과 동일해야 함 (파싱 결과 dataset_from_sweep)
#     MMEA eval yaml 기준 → "UESTC-MMEA-CL"
#   - target_increments: 1.uestc-mmea-mand-herding.yaml → [8, 4, 2]
#   - target_modality: sweep_name의 마지막 (모달리티) 괄호 → 보통 "All"
#   - energy_norm_methods: 이번 sweep에서 energy_norm_method config로 그룹화할 값 목록
#     None이면 데이터에서 unique 값을 자동으로 추출 (4가지 sub-group)
#   - ood_methods: None이면 create_ood_result_table에서 fusion_type에 따라 자동 선택
#
# • cfg: 위 preset에서 선택된 설정 (이후 테이블 생성에 사용)
# • load_sweep_id: None이면 0-3에서 저장한 finished_runs_filename 사용
# =============================================================================

preset = "mmea_mand_wo_morst"  # None(수동) | "mmea_mand" | "mmea_mand_wo_morst"

PRESETS = {
    "mmea_mand": {
        "target_datasets": ["UESTC-MMEA-CL"],
        "target_increments": [8, 4, 2],
        "target_modality": "All",
        "target_seeds": [1993, 1994, 1995, 1996, 1997],
        "target_fusion_types": ["mand_fusion"],
        "cl_methods": ["MAND"],
        "ood_methods": [
            "MaxLogit_Baseline",
            "MaxLogit_Hybrid_UniformSum",
            "MaxLogit_Hybrid_UniformAverage",
        ],
        # energy_norm_method 4가지 sub-group: None이면 데이터에서 자동 추출
        "energy_norm_methods": None,
    },
    "mmea_mand_wo_morst": {
        # Ablation: 2.uestc-mmea-mand-wo-morst-herding.yaml (sweep name에 "Ablation" 포함)
        # parse_sweep_name → model_name_from_sweep = "Ablation"
        "target_datasets": ["UESTC-MMEA-CL"],
        "target_increments": [8, 4, 2],
        "target_modality": "All",
        "target_seeds": [1993, 1994, 1995, 1996, 1997],
        "target_fusion_types": ["mand_fusion"],
        "cl_methods": ["Ablation"],
        "ood_methods": [
            "MaxLogit_Baseline",
            "MaxLogit_Hybrid_UniformSum",
            "MaxLogit_Hybrid_UniformAverage",
        ],
        "energy_norm_methods": None,
    },
    # "dataego_mand": {
    #     "target_datasets": ["DataEgo"], "target_increments": [10, 5, 2], "target_modality": "All",
    #     "target_seeds": [1993, 1994, 1995, 1996, 1997], "target_fusion_types": ["mand_fusion"],
    #     "cl_methods": ["MAND"], "ood_methods": None, "energy_norm_methods": None,
    # },
}

_DEFAULT = "mmea_mand"
cfg = (PRESETS.get(preset, PRESETS[_DEFAULT]) if preset else PRESETS[_DEFAULT]).copy()
load_sweep_id = None
print(f"✅ Preset: {preset or _DEFAULT} (cfg: datasets={cfg['target_datasets']}, inc={cfg['target_increments']})")
print(f"   energy_norm_methods: {cfg.get('energy_norm_methods') or '(auto from data)'}")

---

## 1. OOD 결과 분석 시작

**분석 설정 셀을 먼저 실행**한 뒤, 데이터 로드 → 파싱 → 테이블 생성을 순서대로 실행.


# OOD 결과 테이블 생성 (분석 파이프라인)

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 실행 순서 (반드시 순서대로)
1. **분석 설정** — preset 변경 후 실행하여 cfg 설정
2. **데이터 로드** — CSV에서 df 읽기
3. **Sweep Name 파싱** — model, modality, dataset 컬럼 추가
4. **함수 로드** — create_ood_result_table, save_ood_table_to_excel
5. **테이블 생성 및 저장** — cfg 기반 필터링 → 엑셀 파일 생성

In [7]:
# =============================================================================
# [셀 1. 데이터 로드] CSV에서 Run 데이터 읽기
# =============================================================================
# 파일 경로 우선순위:
#   1) finished_runs_filename (0-3 셀에서 저장한 파일명)
#   2) load_sweep_id가 있으면: finished_runs_data_{load_sweep_id}.csv
#   3) 기본: finished_runs_data.csv
#
# • notebook_dir: notebook/ 폴더 경로 (cwd가 notebook이 아니면 /notebook 추가)
# • df: 로드된 DataFrame. 이후 parse, create_ood_result_table 등에서 사용
# =============================================================================

import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'

# 0-3에서 저장한 파일 사용 (sweep_id 있으면 sweep별 파일, 없으면 기본 파일)
try:
    csv_filename = finished_runs_filename
except NameError:
    try:
        csv_filename = f"finished_runs_data_{load_sweep_id}.csv" if load_sweep_id else "finished_runs_data.csv"
    except NameError:
        csv_filename = "finished_runs_data.csv"
csv_path = notebook_dir / 'results' / csv_filename

print(f"📂 데이터 파일 경로: {csv_path}")
print(f"   파일 존재 여부: {csv_path.exists()}\n")

if not csv_path.exists():
    print("⚠️  파일을 찾을 수 없습니다!")
    print("👉 먼저 셀 0-1, 0-2, 0-3을 실행하여 데이터를 수집하세요.")
else:
    df = pd.read_csv(csv_path)
    print(f"✅ 데이터 로드 완료: {len(df)} runs")
    print(f"   • 컬럼 수: {len(df.columns)}")

📂 데이터 파일 경로: /workspace/MMEA-MoAS/notebook/results/finished_runs_data_xx5mgvsg.csv
   파일 존재 여부: True

✅ 데이터 로드 완료: 15 runs
   • 컬럼 수: 240


In [8]:
# =============================================================================
# [셀 2. Sweep Name 파싱]
# =============================================================================
# sweep_name 형식:
#   (기본)   "Evaluate {model} ({modality}) Method on {dataset} (Seeds ...)"
#   (MAND)   "Evaluate TBN-MAND-Herding (λ=...) (All) Method on UESTC-MMEA-CL (Seeds ..."
#   (신규 A) "Evaluate energy_norm_method=conf_inv(-) TBN-MAND-Herding (λ=...) (All) Method on ..."
#            → sweep name 앞에 "key=value(-) " 형태 접두사 제거
#   (신규 B) "Evaluate energy_norm_method=low_linear/... TBN-MAND-Herding (λ=...) (All) on ..."
#            → "Method" 없이 "on"만 사용, 슬래시로 구분된 접두사 제거
#   (Ablation) "Evaluate Ablation (w/o MoRST) TBN-MAND (All) Method on UESTC-MMEA-CL (Seeds ..."
#            → "Ablation" 이름 안의 "on"을 분리자로 오인하지 않도록 (?:Method on| on ) 패턴 사용
# =============================================================================

def parse_sweep_name(sweep_name):
    if pd.isna(sweep_name):
        return None, None, None
    s = str(sweep_name)
    if "Evaluate " in s:
        s = s[s.index("Evaluate "):]
    # "Method on" 또는 " on " 모두 처리 (단어 중간의 "on" 오인식 방지)
    m_ds = re.search(r"(?:Method on| on )(.+?) \(Seeds", s)
    if not m_ds:
        # 레거시 단일-괄호 형식
        m_old = re.search(r"Evaluate (.+?) \((.+?)\) Method on (.+?) \(Seeds", s)
        if m_old:
            return m_old.group(1).strip(), m_old.group(2).strip(), m_old.group(3).strip()
        return None, None, None
    dataset = m_ds.group(1).strip()
    before = s[:m_ds.start()]
    parens = re.findall(r"\(([^)]+)\)", before)
    modality = parens[-1].strip() if parens else None
    rest = before[len("Evaluate "):] if before.startswith("Evaluate ") else before
    # "key=value(-) " 또는 "key=val1/val2/... " 형태 접두사 제거
    rest = re.sub(r"^(?:\w+=\S+\s+)+", "", rest)
    idx = rest.find(" (")
    model = rest[:idx].strip() if idx >= 0 else rest.strip()
    return model, modality, dataset


df[["model_name_from_sweep", "modality_from_sweep", "dataset_from_sweep"]] = df["sweep_name"].apply(
    lambda x: pd.Series(parse_sweep_name(x))
)

print("✅ Sweep Name 파싱 완료")
print(f"   model   : {df['model_name_from_sweep'].unique()}")
print(f"   modality: {df['modality_from_sweep'].unique()}")
print(f"   dataset : {df['dataset_from_sweep'].unique()}")

In [9]:
# =============================================================================
# [컬럼 목록 확인] df의 모든 컬럼명 출력
# =============================================================================
# • run_id, config, summary, Task/[xx-xx]_acc, Average_OOD/..., model_name_from_sweep 등
# • 테이블 생성 시 사용하는 컬럼 존재 여부 확인용
# =============================================================================
df.keys()


Index(['run_id', 'run_name', 'state', 'sweep_id', 'sweep_name', 'created_at',
       'lr', 'arch', 'host', 'mode',
       ...
       'Task/[18-19]_acc', 'Task/[20-21]_acc', 'Task/[22-23]_acc',
       'Task/[24-25]_acc', 'Task/[26-27]_acc', 'Task/[28-29]_acc',
       'Task/[30-31]_acc', 'model_name_from_sweep', 'modality_from_sweep',
       'dataset_from_sweep'],
      dtype='object', length=243)

In [10]:
# =============================================================================
# [셀 3. 테이블 생성 함수 정의]
# =============================================================================
#
# create_ood_result_table(df, ...):
#   • df를 datasets, increments, modality, seeds, fusion_type, cl_methods,
#     ood_methods, energy_norm_methods로 필터링
#   • energy_norm_method별로 그룹화하여 섹션을 구분
#   • 각 (energy_norm_method, CL_Method, OOD_Method) 조합별로:
#     - ACC: Task/avg_acc → "mean±std" 문자열
#     - Forgetting: Task/forgetting → "mean±std" 문자열
#     - AUC: Average_OOD/{ood_method}_auroc → "mean±std" 문자열
#     - FPR95: Average_OOD/{ood_method}_fpr95 → "mean±std" 문자열
#   • fusion_type이 mand_fusion/auxiliary면 Hybrid 방법도 포함, 아니면 Baseline만
#   • MultiIndex 컬럼: (Dataset (increment: N), Known/Unknown, ACC/Forgetting/AUC/FPR95)
#
# save_ood_table_to_excel(result_df, filename):
#   • result_df를 엑셀로 저장, energy_norm_method 블록별 배경색 구분 + 헤더 스타일링
# =============================================================================

import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.cell.cell import MergedCell
from openpyxl.utils import get_column_letter

def _fmt(series, decimals=2):
    """Series → "mean±std" 문자열. NaN이면 "-" 반환."""
    if len(series) == 0:
        return "-"
    mean = series.mean()
    std  = series.std()
    if pd.isna(mean):
        return "-"
    fmt = f"{{:.{decimals}f}}"
    std_str = fmt.format(std) if not pd.isna(std) else "N/A"
    return f"{fmt.format(mean)}±{std_str}"

def create_ood_result_table(df, datasets=['DataEgo', 'UESTC-MMEA-CL'], increments=[2],
                            modality='All', seeds=[1993, 1994, 1995, 1996, 1997],
                            cl_methods=None, ood_methods=None, fusion_type=None,
                            energy_norm_methods=None):
    '''OOD 결과를 표 형태로 생성 (energy_norm_method 서브그룹 포함, mean±std)'''
    if cl_methods is None:
        cl_methods = ['Replay']

    if ood_methods is None:
        baseline_methods = [
            'MSP_Baseline', 'MaxLogit_Baseline', 'ODIN_Baseline', 'Entropy_Baseline',
            'Energy_Baseline', 'ReAct_Baseline', 'ASH_S_Baseline', 'Scale_Baseline', 'LTS_Baseline'
        ]
        hybrid_methods = [
            'MSP_Hybrid_UniformSum', 'MaxLogit_Hybrid_UniformSum', 'ODIN_Hybrid_UniformSum',
            'Entropy_Hybrid_UniformSum', 'Energy_Hybrid_UniformSum',
            'MSP_Hybrid_UniformAverage', 'MaxLogit_Hybrid_UniformAverage', 'ODIN_Hybrid_UniformAverage',
            'Entropy_Hybrid_UniformAverage', 'Energy_Hybrid_UniformAverage',
            'MSP_Hybrid_ConfRaw', 'MaxLogit_Hybrid_ConfRaw', 'ODIN_Hybrid_ConfRaw',
            'Entropy_Hybrid_ConfRaw', 'Energy_Hybrid_ConfRaw',
            'Entropy_Hybrid_ConfNormalized', 'Energy_Hybrid_ConfNormalized',
        ]
        if fusion_type is not None:
            ft_lower = fusion_type.lower()
            if 'auxiliary' in ft_lower or 'mand_fusion' in ft_lower:
                ood_methods = baseline_methods + hybrid_methods
                print(f"📊 '{fusion_type}' → Baseline + Hybrid ({len(ood_methods)} methods)")
            else:
                ood_methods = baseline_methods
                print(f"📊 '{fusion_type}' → Baseline only ({len(ood_methods)} methods)")
        else:
            ood_methods = baseline_methods + hybrid_methods
            print(f"⚠️  Fusion type not specified → Using all methods ({len(ood_methods)} total)")

    # energy_norm_method 목록 결정
    enm_col = 'energy_norm_method'
    if energy_norm_methods is None:
        if enm_col in df.columns:
            energy_norm_methods = sorted(df[enm_col].dropna().unique().tolist())
            print(f"🔍 energy_norm_methods (auto): {energy_norm_methods}")
        else:
            energy_norm_methods = [None]
            print(f"⚠️  '{enm_col}' 컬럼 없음 → energy_norm_method 그룹화 생략")
    else:
        print(f"🔍 energy_norm_methods (cfg): {energy_norm_methods}")

    results = []
    block_boundaries = []  # (시작 행 인덱스, 끝 행 인덱스, enm 값) - 엑셀 스타일링용

    print("\n" + "=" * 80)
    print("🔍 데이터 필터링 상세 정보")
    print("=" * 80)

    for enm in energy_norm_methods:
        block_start = len(results)

        for cl_method in cl_methods:
            for ood_method in ood_methods:
                row_data = {
                    'energy_norm_method': enm if enm is not None else '-',
                    'CL_Method': cl_method,
                    'OOD_Method': ood_method,
                }

                for dataset in datasets:
                    for increment in increments:
                        filtered = df[
                            (df['model_name_from_sweep'].str.contains(cl_method, na=False)) &
                            (df['modality_from_sweep'] == modality) &
                            (df['dataset_from_sweep'] == dataset)
                        ]
                        if 'increment' in df.columns:
                            filtered = filtered[filtered['increment'] == increment]
                        if 'seed' in df.columns:
                            filtered = filtered[filtered['seed'].isin(seeds)]
                        if fusion_type is not None and 'fusion_type' in df.columns:
                            filtered = filtered[filtered['fusion_type'] == fusion_type]
                        if enm is not None and enm_col in df.columns:
                            filtered = filtered[filtered[enm_col] == enm]

                        enm_label = str(enm) if enm is not None else '-'
                        print(f"📌 ENM={enm_label:20s} | CL={cl_method:12s} | OOD={ood_method:30s} | Inc={increment:2d} → {len(filtered):3d} records")

                        prefix = f"{dataset}_{increment}"

                        acc_col = 'Task/avg_acc'
                        row_data[f'{prefix}_ACC'] = _fmt(filtered[acc_col]) if acc_col in filtered.columns else "-"

                        fgt_col = 'Task/forgetting'
                        row_data[f'{prefix}_Forgetting'] = _fmt(filtered[fgt_col]) if fgt_col in filtered.columns else "-"

                        for metric_name, metric_key in [('AUC', 'auroc'), ('FPR95', 'fpr95')]:
                            col = f'Average_OOD/{ood_method}_{metric_key}'
                            row_data[f'{prefix}_{metric_name}'] = _fmt(filtered[col]) if col in filtered.columns else "-"

                results.append(row_data)

        block_boundaries.append((block_start, len(results) - 1, str(enm) if enm is not None else '-'))

    print("=" * 80 + "\n")

    result_df = pd.DataFrame(results)

    # MultiIndex 컬럼 생성
    KNOWN_METRICS = ('ACC', 'Forgetting')
    UNKNOWN_METRICS = ('AUC', 'FPR95')

    new_columns = []
    for col in result_df.columns:
        if col in ['energy_norm_method', 'CL_Method', 'OOD_Method']:
            new_columns.append(('', '', col))
        else:
            parts = col.rsplit('_', 1)
            if len(parts) == 2:
                dataset_inc, metric = parts[0], parts[1]
                ds_parts = dataset_inc.rsplit('_', 1)
                if len(ds_parts) == 2:
                    ds_name, inc_val = ds_parts[0], ds_parts[1]
                    level0 = f'{ds_name} (increment {inc_val})'
                    level1 = 'Known Classification' if metric in KNOWN_METRICS else ('Unknown Detection' if metric in UNKNOWN_METRICS else '')
                    new_columns.append((level0, level1, metric))
                else:
                    new_columns.append((dataset_inc, '', metric))
            else:
                new_columns.append(('', '', col))

    result_df.columns = pd.MultiIndex.from_tuples(new_columns)
    result_df._block_boundaries = block_boundaries  # 스타일링용 메타데이터
    return result_df


def save_ood_table_to_excel(result_df, filename, apply_formatting=True):

    block_boundaries = getattr(result_df, '_block_boundaries', [])
    result_df.to_excel(filename, merge_cells=True)

    if apply_formatting:
        wb = openpyxl.load_workbook(filename)
        ws = wb.active

        header_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
        header_font = Font(bold=True, size=11)
        thick_border = Border(
            left=Side(style='medium'), right=Side(style='medium'),
            top=Side(style='medium'), bottom=Side(style='medium')
        )
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )
        center = Alignment(horizontal="center", vertical="center")

        n_header_rows = result_df.columns.nlevels
        for row_idx in range(1, n_header_rows + 1):
            for cell in ws[row_idx]:
                if isinstance(cell, MergedCell):
                    continue
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = center
                cell.border = thick_border

        # energy_norm_method 블록별 배경색 교대
        block_colors = ["FFFFFF", "EBF5FB"]          # 흰색 / 연한 파란색
        method_fill_colors = ["E2EFDA", "D5F5E3"]    # 메서드 컬럼 교대

        row_block_map = {}
        for b_idx, (b_start, b_end, _) in enumerate(block_boundaries):
            for r in range(b_start, b_end + 1):
                row_block_map[r] = b_idx

        data_start_row = n_header_rows + 1
        for row in ws.iter_rows(min_row=data_start_row):
            df_row_idx = row[0].row - data_start_row
            b_idx = row_block_map.get(df_row_idx, 0)
            bg_color = block_colors[b_idx % len(block_colors)]
            meth_color = method_fill_colors[b_idx % len(method_fill_colors)]

            for cell in row:
                if isinstance(cell, MergedCell):
                    continue
                cell.alignment = center
                cell.border = thin_border
                if 2 <= cell.column <= 4:  # energy_norm_method, CL_Method, OOD_Method
                    cell.fill = PatternFill(start_color=meth_color, end_color=meth_color, fill_type="solid")
                else:
                    cell.fill = PatternFill(start_color=bg_color, end_color=bg_color, fill_type="solid")

        # 열 너비
        ws.column_dimensions['A'].width = 5
        ws.column_dimensions['B'].width = 22   # energy_norm_method
        ws.column_dimensions['C'].width = 12   # CL_Method
        ws.column_dimensions['D'].width = 18   # OOD_Method
        for col_idx in range(5, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col_idx)].width = 20

        wb.save(filename)

    print(f"✅ 엑셀 저장 완료: {filename}")
    print(f"   • {result_df.shape[0]} rows × {result_df.shape[1]} columns")

print("✅ 함수 로드 완료")

✅ 함수 로드 완료


In [11]:
# =============================================================================
# [셀 4. 테이블 생성 및 저장]
# =============================================================================
# • 파일명: ood_results_{dataset}_{project_short}_sweep-{sweep_id}_inc{N}_{modality}_{fusion_type}.xlsx
#   - dataset     : target_datasets 조합
#   - project_short: project_name 마지막 슬래시 이후 (최대 30자, 특수문자 → '-')
#   - sweep_id    : 셀 0-1의 sweep_id (없으면 "nosweep")
# • fusion_type × increment 조합마다:
#   1) create_ood_result_table() 호출 → ood_table (energy_norm_method 그룹화 포함)
#   2) display(ood_table.head(10)) 미리보기
#   3) save_ood_table_to_excel()로 엑셀 저장
# =============================================================================

import re as _re

target_datasets     = cfg['target_datasets']
target_increments   = cfg['target_increments']
target_modality     = cfg['target_modality']
target_seeds        = cfg['target_seeds']
target_fusion_types = cfg['target_fusion_types']
cl_methods          = cfg['cl_methods']
ood_methods         = cfg.get('ood_methods')
energy_norm_methods = cfg.get('energy_norm_methods')  # None이면 데이터에서 자동 추출

output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

# 파일명 구성 재료
try:
    _sweep_id_for_fn = sweep_id if sweep_id else "nosweep"
except NameError:
    _sweep_id_for_fn = f"{load_sweep_id}" if load_sweep_id else "nosweep"

try:
    _proj_short = _re.sub(r'[^\w-]', '-', project_name.split('/')[-1])[:30]
except NameError:
    _proj_short = "proj"

dataset_str = '-'.join(target_datasets)

print("=" * 100)
print("🔄 OOD 결과 테이블 생성")
print("=" * 100)
print(f"  • 데이터셋:  {target_datasets}")
print(f"  • Increment: {target_increments}")
print(f"  • Fusion:    {target_fusion_types}")
print(f"  • CL:        {cl_methods}")
print(f"  • Sweep ID:  {_sweep_id_for_fn}")
print(f"  • Project:   {_proj_short}")
print()

for fusion_type in target_fusion_types:
    for increment in target_increments:
        print("\n" + "=" * 100)
        print(f"📊 {fusion_type}, inc={increment}")
        print("=" * 100)

        output_filename = (
            output_dir /
            f"ood_results_{dataset_str}_{_proj_short}_sweep-{_sweep_id_for_fn}"
            f"_inc{increment}_{target_modality}_{fusion_type}.xlsx"
        )

        ood_table = create_ood_result_table(
            df=df,
            datasets=target_datasets,
            increments=[increment],
            modality=target_modality,
            seeds=target_seeds,
            fusion_type=fusion_type,
            cl_methods=cl_methods,
            ood_methods=ood_methods,
            energy_norm_methods=energy_norm_methods,
        )

        print(f"\n✅ 테이블 생성 완료: {ood_table.shape[0]} rows × {ood_table.shape[1]} columns")
        print("\n📋 테이블 미리보기 (처음 10행)")
        print("-" * 100)
        display(ood_table.head(10))

        print("\n💾 엑셀 파일로 저장 중...")
        save_ood_table_to_excel(ood_table, str(output_filename), apply_formatting=True)

        print(f"✅ 완료! 파일: {output_filename}")

print("\n" + "=" * 100)
print(f"🎉 모든 작업 완료! 총 {len(target_fusion_types) * len(target_increments)}개의 엑셀 파일이 생성되었습니다.")
print("=" * 100)


🔄 OOD 결과 테이블 생성
  • 데이터셋:  ['UESTC-MMEA-CL']
  • Increment: [8, 4, 2]
  • Fusion:    ['mand_fusion']
  • CL:        ['MAND']
  • Sweep ID:  xx5mgvsg
  • Project:   Experimental-Results-on-the-MM


📊 mand_fusion, inc=8
🔍 energy_norm_methods (auto): ['zscore']

🔍 데이터 필터링 상세 정보
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Baseline              | Inc= 8 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Inc= 8 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Inc= 8 →   0 records


✅ 테이블 생성 완료: 3 rows × 7 columns

📋 테이블 미리보기 (처음 10행)
----------------------------------------------------------------------------------------------------


/tmp/ipykernel_2702855/2089789520.py:165: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  result_df._block_boundaries = block_boundaries  # 스타일링용 메타데이터


\
                                                                 
  energy_norm_method CL_Method                      OOD_Method   
0             zscore      MAND               MaxLogit_Baseline   
1             zscore      MAND      MaxLogit_Hybrid_UniformSum   
2             zscore      MAND  MaxLogit_Hybrid_UniformAverage   

  UESTC-MMEA-CL (increment 8)                                     
         Known Classification            Unknown Detection        
                          ACC Forgetting               AUC FPR95  
0                           -          -                 -     -  
1                           -          -                 -     -  
2                           -          -                 -     -


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc8_All_mand_fusion.xlsx
   • 3 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc8_All_mand_fusion.xlsx

📊 mand_fusion, inc=4
🔍 energy_norm_methods (auto): ['zscore']

🔍 데이터 필터링 상세 정보
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Baseline              | Inc= 4 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Inc= 4 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Inc= 4 →   0 records


✅ 테이블 생성 완료: 3 rows × 7 columns

📋 테이블 미리보기 (처음 10행)
----------------------------------------------------------------------------------------------------


/tmp/ipykernel_2702855/2089789520.py:165: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  result_df._block_boundaries = block_boundaries  # 스타일링용 메타데이터


\
                                                                 
  energy_norm_method CL_Method                      OOD_Method   
0             zscore      MAND               MaxLogit_Baseline   
1             zscore      MAND      MaxLogit_Hybrid_UniformSum   
2             zscore      MAND  MaxLogit_Hybrid_UniformAverage   

  UESTC-MMEA-CL (increment 4)                                     
         Known Classification            Unknown Detection        
                          ACC Forgetting               AUC FPR95  
0                           -          -                 -     -  
1                           -          -                 -     -  
2                           -          -                 -     -


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc4_All_mand_fusion.xlsx
   • 3 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc4_All_mand_fusion.xlsx

📊 mand_fusion, inc=2
🔍 energy_norm_methods (auto): ['zscore']

🔍 데이터 필터링 상세 정보
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Baseline              | Inc= 2 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Inc= 2 →   0 records
📌 ENM=zscore               | CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Inc= 2 →   0 records


✅ 테이블 생성 완료: 3 rows × 7 columns

📋 테이블 미리보기 (처음 10행)
----------------------------------------------------------------------------------------------------


/tmp/ipykernel_2702855/2089789520.py:165: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  result_df._block_boundaries = block_boundaries  # 스타일링용 메타데이터


\
                                                                 
  energy_norm_method CL_Method                      OOD_Method   
0             zscore      MAND               MaxLogit_Baseline   
1             zscore      MAND      MaxLogit_Hybrid_UniformSum   
2             zscore      MAND  MaxLogit_Hybrid_UniformAverage   

  UESTC-MMEA-CL (increment 2)                                     
         Known Classification            Unknown Detection        
                          ACC Forgetting               AUC FPR95  
0                           -          -                 -     -  
1                           -          -                 -     -  
2                           -          -                 -     -


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc2_All_mand_fusion.xlsx
   • 3 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-MoAS/notebook/results/ood_results_UESTC-MMEA-CL_Experimental-Results-on-the-MM_sweep-xx5mgvsg_inc2_All_mand_fusion.xlsx

🎉 모든 작업 완료! 총 3개의 엑셀 파일이 생성되었습니다.


In [12]:
# =============================================================================
# [데이터 진단] cfg 설정에 맞는 데이터가 실제로 있는지 확인
# =============================================================================
# • 1) increment 분포: df에 몇 개의 increment 값이 있는지
# • 2) fusion_type 분포: mand_fusion, concat 등
# • 3) cfg 조건별 record 수: (CL, Dataset, inc, fusion_type) 조합마다 몇 건인지
#   → 0건이면 테이블에 NaN으로 나오므로, sweep/실험 설정 확인 필요
# =============================================================================

print("=" * 100)
print(f"🔍 데이터 진단 (preset={preset}, fusion={cfg['target_fusion_types']}, inc={cfg['target_increments']})")
print("=" * 100)

print("\n1️⃣ increment 분포:")
print(df['increment'].value_counts().sort_index() if 'increment' in df.columns else "(컬럼 없음)")

print("\n2️⃣ fusion_type 분포:")
print(df['fusion_type'].value_counts() if 'fusion_type' in df.columns else "(컬럼 없음)")

print("\n3️⃣ 현재 cfg 조건에 맞는 데이터:")
req_cols = ['increment', 'fusion_type', 'model_name_from_sweep', 'modality_from_sweep', 'dataset_from_sweep']
if all(c in df.columns for c in req_cols):
    for ft in cfg['target_fusion_types']:
        for inc in cfg['target_increments']:
            for cl in cfg['cl_methods']:
                for ds in cfg['target_datasets']:
                    f = df[(df['increment']==inc) & (df['fusion_type']==ft) &
                           (df['model_name_from_sweep'].str.contains(cl, na=False)) &
                           (df['modality_from_sweep']==cfg['target_modality']) &
                           (df['dataset_from_sweep']==ds)]
                    print(f"   {cl:8s} | {ds:18s} | inc={inc:2d} | {ft:18s} → {len(f):3d} records")
else:
    print("   (필요 컬럼 부족)")

print("\n" + "=" * 100)

🔍 데이터 진단 (preset=mmea_mand, fusion=['mand_fusion'], inc=[8, 4, 2])

1️⃣ increment 분포:
2    5
4    5
8    5
Name: increment, dtype: int64

2️⃣ fusion_type 분포:
mand_fusion    15
Name: fusion_type, dtype: int64

3️⃣ 현재 cfg 조건에 맞는 데이터:
   MAND     | UESTC-MMEA-CL      | inc= 8 | mand_fusion        →   0 records
   MAND     | UESTC-MMEA-CL      | inc= 4 | mand_fusion        →   0 records
   MAND     | UESTC-MMEA-CL      | inc= 2 | mand_fusion        →   0 records



In [13]:
# =============================================================================
# [추가 확인] sweep_name 파싱 결과 - model_name_from_sweep 고유값
# =============================================================================
# • parse_sweep_name으로 추출된 모델명 목록 (예: TBN-MAND, TBN-Replay 등)
# • cl_methods 필터와 일치하는지 확인용
# =============================================================================
df['model_name_from_sweep'].unique()

array(['Ablati'], dtype=object)